In [3]:
import time
import threading
import requests
from pynvml import *


# --------------------------------------------------
# 1. Read GPU power
# --------------------------------------------------

nvmlInit()

handle = nvmlDeviceGetHandleByIndex(0)


def get_power_watts():
    """
    Returns current GPU power in watts.
    """
    return nvmlDeviceGetPowerUsage(handle) / 1000.0


# --------------------------------------------------
# 2. Measure idle power
# --------------------------------------------------

def measure_idle_power(seconds=3):
    """
    Measures average GPU power while idle.
    """

    readings = []

    start = time.perf_counter()

    while time.perf_counter() - start < seconds:
        readings.append(get_power_watts())
        time.sleep(0.05)

    return sum(readings) / len(readings)


# --------------------------------------------------
# 3. Background power monitor
# --------------------------------------------------

def power_monitor(samples, stop_event):
    """
    Records GPU power every 50ms.
    """

    while not stop_event.is_set():

        timestamp = time.perf_counter()
        power = get_power_watts()

        samples.append((timestamp, power))
        print(f"GPU Power: {power:.2f} W")

        time.sleep(0.05)


# --------------------------------------------------
# 4. Calculate energy using trapezoidal rule
# --------------------------------------------------

def calculate_energy(samples):
    """
    Calculates energy in joules.

    Energy = integral of Power over Time
    """

    energy = 0.0

    for i in range(1, len(samples)):

        t1, p1 = samples[i - 1]
        t2, p2 = samples[i]

        dt = t2 - t1

        # Trapezoidal rule
        energy += ((p1 + p2) / 2) * dt

    return energy


# --------------------------------------------------
# 5. Run AI + measure energy
# --------------------------------------------------

def run_and_measure(prompt, model):

    # A. Measure idle power
    print("Measuring idle power...")

    P_idle = measure_idle_power(seconds=3)

    print(f"Idle power: {P_idle:.2f} W")


    # B. Start power monitoring
    samples = []

    stop_event = threading.Event()

    monitor_thread = threading.Thread(
        target=power_monitor,
        args=(samples, stop_event)
    )

    monitor_thread.start()


    # C. Send request to Ollama
    start_time = time.perf_counter()

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )

    end_time = time.perf_counter()


    # D. Stop monitoring
    stop_event.set()
    monitor_thread.join()


    # E. Calculate latency
    latency = end_time - start_time


    # F. Calculate total energy
    total_energy = calculate_energy(samples)


    # G. Calculate idle energy
    idle_energy = P_idle * latency


    # H. Calculate net dynamic energy
    net_energy = total_energy - idle_energy


    return response.json()["response"], latency, net_energy


# --------------------------------------------------
# 6. Example
# --------------------------------------------------

if __name__ == "__main__":

    response, latency, energy = run_and_measure(
        "Explain machine learning in very simple words.",
        "hf.co/mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated-GGUF:Q4_K_M"
    )

    print("\nResponse:")
    print(response)

    print(f"\nLatency: {latency:.2f} seconds")
    print(f"Energy: {energy:.4f} joules")


    nvmlShutdown()

Measuring idle power...
Idle power: 52.69 W
GPU Power: 69.36 W
GPU Power: 69.36 W
GPU Power: 69.36 W
GPU Power: 69.36 W
GPU Power: 69.36 W
GPU Power: 69.36 W
GPU Power: 69.36 W
GPU Power: 69.30 W
GPU Power: 69.30 W
GPU Power: 69.30 W
GPU Power: 69.30 W
GPU Power: 69.30 W
GPU Power: 69.30 W
GPU Power: 69.30 W
GPU Power: 69.30 W
GPU Power: 69.08 W
GPU Power: 69.08 W
GPU Power: 69.08 W
GPU Power: 69.08 W
GPU Power: 69.08 W
GPU Power: 69.08 W
GPU Power: 69.08 W
GPU Power: 69.08 W
GPU Power: 55.08 W
GPU Power: 55.08 W
GPU Power: 55.08 W
GPU Power: 55.08 W
GPU Power: 55.08 W
GPU Power: 55.08 W
GPU Power: 55.08 W
GPU Power: 55.08 W
GPU Power: 33.37 W
GPU Power: 33.37 W
GPU Power: 33.37 W
GPU Power: 33.37 W
GPU Power: 33.37 W
GPU Power: 33.37 W
GPU Power: 33.37 W
GPU Power: 33.37 W
GPU Power: 38.02 W
GPU Power: 38.02 W
GPU Power: 38.02 W
GPU Power: 38.02 W
GPU Power: 38.02 W
GPU Power: 38.02 W
GPU Power: 38.02 W
GPU Power: 38.02 W
GPU Power: 131.16 W
GPU Power: 131.16 W
GPU Power: 131.16 W
GPU